# V11.2: Temporal + Fiber Field Reconstruction

## Core Principle

A token is not a dense vector. It is **many sparse activations firing simultaneously**
across decoupled fiber channels — point sources of potential on the manifold. The
model's job: solve for the **full field** those sources induce on a context-warped
geometry, then settle that field back into a clean sparse attractor.

## Forward Pass

```
token → sparse sources (25% active per subbundle) →
  [ Wilson line (multi-scale temporal: local+global, in/out projections) →
    field reconstruction:
      stage 1: temporal mixing (causal exp conv across positions) →
      stage 2: fiber-dim FFT (fill unknowns, context-warped D & A) →
    memory routing (from dense field) →
    Langevin settling (field → sparse, adaptive proximal every step) →
    residual + FFN →
    re-sparsify (per-subbundle top-k) ] × N →
  final_norm → decode
```

## What Changed from v11.1

| v11.1 | v11.2 |
|---|---|
| Field recon = fiber-dim FFT only (no temporal) | Field recon = temporal conv THEN fiber-dim FFT |
| Wilson line = input proj only | Wilson line = input proj + output proj (remixes scales) |
| 2 causal convolutions per block | 3 causal convolutions per block |
| Lost temporal mixing → slower learning | Temporal mixing restored + fiber reconstruction kept |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
import math
import os
import urllib.request
from tqdm.notebook import tqdm

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# device = torch.device("cpu")
print(f"Device: {device}")

data_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "v8_real_text", "data")
data_path = os.path.join(data_dir, "input.txt")

if not os.path.exists(data_path):
    os.makedirs(data_dir, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
def encode(s): return [stoi[c] for c in s]
def decode(t): return "".join(itos[i] for i in t if i in itos)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]
print(f"Train: {len(train_data):,} | Val: {len(val_data):,} chars")
print(f"Vocab: {vocab_size} unique characters (was 256 ASCII)")

Device: mps
Train: 1,003,854 | Val: 111,540 chars
Vocab: 65 unique characters (was 256 ASCII)


In [2]:
@dataclass
class V11Config:
    fiber_dim: int = 512
    n_subbundles: int = 8
    context_dim: int = 128

    vocab_size: int = 65  # overridden below from actual data
    max_seq_len: int = 1024

    atoms_per_subbundle: int = 192
    k_active: int = 16
    topk_per_sub: int = 16   # 16/64 = 25% active, 75% sparse

    n_blocks: int = 4
    langevin_steps: int = 5
    langevin_lr: float = 0.1
    beta_init: float = 1.0
    beta_final: float = 12.0
    sparsity_lambda: float = 0.05

    learning_rate: float = 3e-4
    warmup_steps: int = 2000
    dropout: float = 0.1
    batch_size: int = 8
    seq_len: int = 512
    max_steps: int = 20000
    eval_interval: int = 500
    eval_steps: int = 10

    @property
    def subbundle_dim(self):
        return self.fiber_dim // self.n_subbundles

config = V11Config(vocab_size=vocab_size)
print(f"Vocab: {config.vocab_size} (from data)")
print(f"Fiber: {config.fiber_dim} = {config.n_subbundles} x {config.subbundle_dim}")
print(f"Sparsity: {config.topk_per_sub}/{config.subbundle_dim} active per subbundle "
      f"= {100*config.topk_per_sub/config.subbundle_dim:.0f}% "
      f"({config.topk_per_sub * config.n_subbundles}/{config.fiber_dim} total sources)")
print(f"Atoms: {config.atoms_per_subbundle} per subbundle, k={config.k_active} active")
print(f"Langevin: {config.langevin_steps} steps, proximal at EVERY step")
print(f"LR: {config.learning_rate} with {config.warmup_steps}-step cosine warmup")

def get_batch(split_data, cfg):
    max_start = len(split_data) - cfg.seq_len - 1
    starts = torch.randint(0, max_start, (cfg.batch_size,))
    return torch.stack([split_data[s:s+cfg.seq_len] for s in starts]).to(device)

Vocab: 65 (from data)
Fiber: 512 = 8 x 64
Sparsity: 16/64 active per subbundle = 25% (128/512 total sources)
Atoms: 192 per subbundle, k=16 active
Langevin: 5 steps, proximal at EVERY step
LR: 0.0003 with 2000-step cosine warmup


In [3]:
# ── Shared utility ──────────────────────────────────────────────────

def causal_exp_conv(x, log_decay_params):
    """Causal FFT convolution with per-dim exponential decay kernel.
    Used by both Wilson line (context accumulation) and field reconstructor
    (heat kernel propagation). O(T log T) via FFT."""
    B, T, D = x.shape
    decay = F.softplus(log_decay_params)
    lags = torch.arange(T, device=x.device).float()
    kernel = torch.exp(-decay.unsqueeze(0) * lags.unsqueeze(-1))
    kernel = kernel / (kernel.sum(0, keepdim=True) + 1e-8)

    pad_len = T
    x_p = F.pad(x, (0, 0, 0, pad_len))
    k_p = F.pad(kernel, (0, 0, 0, pad_len))
    X = torch.fft.fft(x_p, dim=1)
    K = torch.fft.fft(k_p, dim=0).unsqueeze(0)
    return torch.fft.ifft(X * K, dim=1).real[:, :T, :]


def apply_gauge_rotation(x, phases):
    """U(1) gauge rotation on pairs of fiber dimensions.
    Implements exp(-iw int A) from Architecture.md in real arithmetic."""
    x_even, x_odd = x[..., 0::2], x[..., 1::2]
    cos_p, sin_p = torch.cos(phases), torch.sin(phases)
    out = torch.zeros_like(x)
    out[..., 0::2] = cos_p * x_even - sin_p * x_odd
    out[..., 1::2] = sin_p * x_even + cos_p * x_odd
    return out


def differentiable_proximal(x, thr):
    """Soft-thresholding with correct gradient flow.
    Forward: sign(x) * max(|x| - thr, 0)  (standard proximal)
    Backward: gradient = 1 where |x| > thr, 0 elsewhere.

    torch.sign() has zero gradient everywhere in autograd, which kills
    upstream gradients. This rewrite avoids sign() in the gradient path:
      (x - sign(x).detach() * thr) * mask
    where mask = (|x| > thr). The mask is a step function (zero grad from
    autograd), so grad flows only through x, gated by the mask."""
    mask = (x.abs() > thr).float()
    return (x - x.sign().detach() * thr) * mask


def subbundle_resparsify(x, cfg):
    """Per-subbundle top-k to restore sparse invariant between blocks.
    Vectorized: reshapes to (*, K, sd), does one topk, reshapes back."""
    shape = x.shape
    x_subs = x.reshape(*shape[:-1], cfg.n_subbundles, cfg.subbundle_dim)
    _, idx = torch.topk(x_subs.abs(), cfg.topk_per_sub, dim=-1)
    mask = torch.zeros_like(x_subs)
    mask.scatter_(-1, idx, 1.0)
    return (x_subs * mask).reshape(shape)


# ── Embedding ───────────────────────────────────────────────────────

class SparseSourceEmbedding(nn.Module):
    """Token + position -> sparse source terms on the fiber bundle.
    Active dimensions are source terms. Inactive dimensions are unknowns
    to be estimated by the field reconstruction."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)

        pe = torch.zeros(cfg.max_seq_len, cfg.fiber_dim)
        pos = torch.arange(cfg.max_seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, cfg.fiber_dim, 2).float()
                        * (-math.log(10000.0) / cfg.fiber_dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, token_ids):
        B, T = token_ids.shape
        cfg = self.cfg
        x = self.embedding(token_ids) + self.pe[:, :T, :]

        x_subs = x.reshape(B, T, cfg.n_subbundles, cfg.subbundle_dim)
        _, idx = torch.topk(x_subs.abs(), cfg.topk_per_sub, dim=-1)
        mask = torch.zeros_like(x_subs)
        mask.scatter_(-1, idx, 1.0)
        return (x_subs * mask).reshape(B, T, -1), mask.reshape(B, T, -1)


# ── Context-dependent geometry ──────────────────────────────────────

class WilsonLineAccumulator(nn.Module):
    """Multi-scale manifold accumulator for temporal structure.

    Input projection rotates sparse sources into the manifold's coordinate
    frame. Dual-scale causal convolution gives the geometry both local
    (fast-decay) and global (slow-decay) temporal range. Output projection
    remixes the scales so the manifold can learn which combinations of
    local and global context matter at each fiber dimension.

    No gates — the learned geometry itself determines how meaning
    transitions between token positions."""

    def __init__(self, cfg):
        super().__init__()
        self.proj_in = nn.Linear(cfg.fiber_dim, cfg.fiber_dim)
        half = cfg.fiber_dim // 2
        self.log_decay_local = nn.Parameter(torch.linspace(0.5, 3.0, half))
        self.log_decay_global = nn.Parameter(torch.linspace(-1.5, 0.5, half))
        self.proj_out = nn.Linear(cfg.fiber_dim, cfg.fiber_dim)
        self.norm = nn.LayerNorm(cfg.fiber_dim)

    def forward(self, x):
        h = self.proj_in(x)
        h_local, h_global = h.chunk(2, dim=-1)
        local_ctx = causal_exp_conv(h_local, self.log_decay_local)
        global_ctx = causal_exp_conv(h_global, self.log_decay_global)
        return self.norm(self.proj_out(torch.cat([local_ctx, global_ctx], dim=-1)))


# ── Field reconstruction (fiber-dimension diffusion) ────────────────

class FiberFieldReconstructor(nn.Module):
    """Two-stage field reconstruction from sparse sources:

    Stage 1 — TEMPORAL MIXING (causal conv along dim=1):
      Spreads sparse sources across positions. Each position receives a
      weighted history of all prior positions via exponential decay.
      Fast-decay dims capture local context (bigrams, trigrams).
      Slow-decay dims capture long-range context (speaker, topic).
      This is the heat equation running along the sequence.

    Stage 2 — FIBER RECONSTRUCTION (per-subbundle FFT along dim=-1):
      Fills in unknown fiber dimensions from the temporally-mixed signal.
      The sparse peaks have broad spectral content; context-dependent
      diffusion D dampens high frequencies, smoothing peaks into
      neighboring dimensions. Gauge connection A rotates the phase,
      warping reconstruction based on manifold geometry.

    Both stages are essential: temporal mixing gives each position
    information about its past; fiber reconstruction fills in the
    75% of dimensions that are unknown from that enriched signal."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        # Stage 1: temporal mixing across positions
        self.temporal_log_decay = nn.Parameter(
            torch.linspace(-1.0, 2.0, cfg.fiber_dim))
        # Stage 2: fiber reconstruction with context-dependent coefficients
        self.D_proj = nn.Linear(cfg.fiber_dim, cfg.fiber_dim)
        self.A_proj = nn.Linear(cfg.fiber_dim, cfg.fiber_dim)
        self.norm = nn.LayerNorm(cfg.fiber_dim)

    def forward(self, sparse_sources, context):
        cfg = self.cfg
        B, T, D = sparse_sources.shape
        sd = cfg.subbundle_dim
        K = cfg.n_subbundles

        # Stage 1: temporal mixing — causal conv spreads sources across positions
        mixed = causal_exp_conv(sparse_sources, self.temporal_log_decay)

        # Stage 2: fiber reconstruction — vectorized across all subbundles
        diffusion_coeff = F.softplus(self.D_proj(context)).reshape(B, T, K, sd)
        gauge_connection = self.A_proj(context).reshape(B, T, K, sd)
        mixed_subs = mixed.reshape(B, T, K, sd)

        freqs = torch.fft.fftfreq(sd, d=1.0, device=sparse_sources.device)
        X_w = torch.fft.fft(mixed_subs, dim=-1)
        decay = torch.exp(-diffusion_coeff * freqs ** 2)
        phase = torch.exp(-1j * freqs * gauge_connection)
        field = torch.fft.ifft(X_w * decay * phase, dim=-1).real.reshape(B, T, D)

        return self.norm(field)


# ── Memory bank ─────────────────────────────────────────────────────

class MemoryBank(nn.Module):
    """Dynamic memory bank routed by the reconstructed FIELD (not raw sources).
    The dense field provides richer context for atom selection.

    Uses score-weighted atoms: softmax over routing logits before topk,
    then selected atoms are scaled by their (differentiable) routing
    scores. This creates gradient flow through the router — without it,
    topk indices are non-differentiable and the router never learns."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        sd, K, A = cfg.subbundle_dim, cfg.n_subbundles, cfg.atoms_per_subbundle
        self.dictionaries = nn.ParameterList(
            [nn.Parameter(torch.randn(A, sd) * 0.02) for _ in range(K)]
        )
        self.context_proj = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.context_dim), nn.SiLU()
        )
        self.routers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.context_dim + sd, A), nn.SiLU(), nn.Linear(A, A)
            ) for _ in range(K)
        ])

    def forward(self, field, sparse_local):
        cfg = self.cfg
        sd = cfg.subbundle_dim
        B, T, _ = field.shape
        ctx = self.context_proj(field).reshape(B * T, -1)
        x_flat = sparse_local.reshape(B * T, -1)
        x_chunks = x_flat.split(sd, dim=-1)

        M_list = []
        for chunk, dic, router in zip(x_chunks, self.dictionaries, self.routers):
            D_n = F.normalize(dic, dim=-1)
            logits = router(torch.cat([ctx, chunk], dim=-1))
            # Softmax before topk: scores are differentiable w.r.t. logits.
            # topk on scores returns (values, indices) where values have grad.
            scores = F.softmax(logits, dim=-1)
            topk_scores, idx = torch.topk(scores, cfg.k_active, dim=-1)
            # Scale atoms by routing scores — gradient flows through topk_scores
            # to logits to router to context_proj to field.
            M_k = D_n[idx] * topk_scores.unsqueeze(-1)
            M_list.append(M_k)
        return M_list


# ── Langevin settling (reverse diffusion) ───────────────────────────

class LangevinFieldSettler(nn.Module):
    """Reverse diffusion: dense field -> sparse attractor.
    CRITICAL: initialized from the FIELD, not raw sparse input.
    Proximal soft-thresholding at EVERY step with differentiable gradient.
    No noise during training."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg

    def forward(self, field, M_list):
        cfg = self.cfg
        B, T, D = field.shape
        sd = cfg.subbundle_dim
        K = cfg.n_subbundles
        BT = B * T
        x = field  # no clone needed — we overwrite with arithmetic

        betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps,
                               device=field.device)
        # Stack all subbundle memory banks: list of (BT,k,sd) -> (BT,K,k,sd)
        M_all = torch.stack(M_list, dim=1)

        for step in range(cfg.langevin_steps):
            beta = betas[step].item()

            # Vectorized Hopfield gradient across all K subbundles at once
            x_subs = x.reshape(BT, K, sd)
            sim = torch.einsum('bks,bkas->bka', x_subs, M_all)
            w = F.softmax(beta * sim, dim=-1)
            grad = -torch.einsum('bka,bkas->bks', w, M_all)
            x = x - cfg.langevin_lr * grad.reshape(B, T, D)

            if not self.training:
                x = x + math.sqrt(2.0 * cfg.langevin_lr / beta) * torch.randn_like(x)

            # Vectorized adaptive proximal across all subbundles
            if cfg.sparsity_lambda > 0:
                x_subs = x.reshape(B, T, K, sd)
                rms = x_subs.norm(dim=-1, keepdim=True) / math.sqrt(sd)
                thr = cfg.sparsity_lambda * rms
                x = differentiable_proximal(x_subs, thr).reshape(B, T, D)

        return x


# ── FFN ─────────────────────────────────────────────────────────────

class FiberFFN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = cfg.fiber_dim * 2
        self.net = nn.Sequential(
            nn.Linear(cfg.fiber_dim, h), nn.SiLU(), nn.Dropout(cfg.dropout),
            nn.Linear(h, cfg.fiber_dim), nn.Dropout(cfg.dropout),
        )
    def forward(self, x):
        return self.net(x)


# ── Geometric block ─────────────────────────────────────────────────

class GeometricBlock(nn.Module):
    """One forward-reverse diffusion cycle:
    sparse → wilson(temporal) → fiber_reconstruct(field) → route → settle → FFN → re-sparsify"""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.norm1 = nn.LayerNorm(cfg.fiber_dim)
        self.norm2 = nn.LayerNorm(cfg.fiber_dim)
        self.wilson = WilsonLineAccumulator(cfg)
        self.reconstructor = FiberFieldReconstructor(cfg)
        self.memory = MemoryBank(cfg)
        self.settler = LangevinFieldSettler(cfg)
        self.ffn = FiberFFN(cfg)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, sparse_sources):
        h = self.norm1(sparse_sources)

        # Wilson line: manifold accumulates temporal context
        context = self.wilson(h)

        # Forward diffusion along FIBER dimension:
        # sparse sources → dense field (fills in unknown dimensions)
        field = self.reconstructor(h, context)

        # Memory routing from reconstructed field (richer than raw sparse)
        M_list = self.memory(field, h)

        # Reverse diffusion: field → sparse attractor
        settled = self.settler(field, M_list)

        x = sparse_sources + self.dropout(settled)
        x = x + self.ffn(self.norm2(x))

        # Re-sparsify: maintain the sparse source invariant for the next block
        x = subbundle_resparsify(x, self.cfg)
        return x


# ── Full model ──────────────────────────────────────────────────────

class SparseFieldCLM(nn.Module):
    """V11: Sparse Source Field Reconstruction CLM."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embedding = SparseSourceEmbedding(cfg)
        self.blocks = nn.ModuleList([GeometricBlock(cfg) for _ in range(cfg.n_blocks)])
        self.final_norm = nn.LayerNorm(cfg.fiber_dim)
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim), nn.SiLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )
        self.register_buffer("block_loss_weights", torch.linspace(0.1, 1.0, cfg.n_blocks))

    def forward(self, token_ids, return_repr=False):
        B, T = token_ids.shape
        x, support_mask = self.embedding(token_ids)

        intermediate_logits = []
        for block in self.blocks:
            x = block(x)
            intermediate_logits.append(self.decoder(self.final_norm(x))[:, :-1, :])

        info = {
            "mean_sparsity": (x.abs() < 1e-3).float().mean().item(),
            "intermediate_logits": intermediate_logits,
            "block_loss_weights": self.block_loss_weights,
            "support_mask": support_mask,
        }
        if return_repr:
            info["final_repr"] = x
        return intermediate_logits[-1], info


model = SparseFieldCLM(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_embed = sum(p.numel() for p in model.embedding.parameters())
n_wilson = sum(sum(p.numel() for p in blk.wilson.parameters()) for blk in model.blocks)
n_recon = sum(sum(p.numel() for p in blk.reconstructor.parameters()) for blk in model.blocks)
n_memory = sum(sum(p.numel() for p in blk.memory.parameters()) for blk in model.blocks)
n_ffn = sum(sum(p.numel() for p in blk.ffn.parameters()) for blk in model.blocks)
n_decoder = sum(p.numel() for p in model.decoder.parameters()) + \
            sum(p.numel() for p in model.final_norm.parameters())
n_other = n_params - n_embed - n_wilson - n_recon - n_memory - n_ffn - n_decoder

print(f"Total parameters: {n_params:,}")
print(f"  Embedding:                    {n_embed:,} ({100*n_embed/n_params:.1f}%)")
print(f"  Wilson lines (4, multi-scale): {n_wilson:,} ({100*n_wilson/n_params:.1f}%)")
print(f"  Fiber reconstructors (4):      {n_recon:,} ({100*n_recon/n_params:.1f}%)")
print(f"  Memory banks (4):              {n_memory:,} ({100*n_memory/n_params:.1f}%)")
print(f"  FFN (4):                       {n_ffn:,} ({100*n_ffn/n_params:.1f}%)")
print(f"  Decoder + norms:               {n_decoder:,} ({100*n_decoder/n_params:.1f}%)")
print(f"  Block norms:                   {n_other:,} ({100*n_other/n_params:.1f}%)")
print(f"\nv11.1 was ~10.9M | v11.0 was 9.3M | nanoGPT ~10M → CE 1.47")
print(f"\nKey changes from v11.1:")
print(f"  Wilson line: added output proj ({n_wilson:,} params, remixes local+global)")
print(f"  Field recon: temporal mixing THEN fiber-dim FFT (both stages)")
print(f"  3 causal convolutions per block: Wilson local, Wilson global, field temporal")

Total parameters: 11,781,185
  Embedding:                    33,280 (0.3%)
  Wilson lines (4, multi-scale): 2,107,392 (17.9%)
  Fiber reconstructors (4):      2,107,392 (17.9%)
  Memory banks (4):              3,027,456 (25.7%)
  FFN (4):                       4,200,448 (35.7%)
  Decoder + norms:               297,025 (2.5%)
  Block norms:                   8,192 (0.1%)

v11.1 was ~10.9M | v11.0 was 9.3M | nanoGPT ~10M → CE 1.47

Key changes from v11.1:
  Wilson line: added output proj (2,107,392 params, remixes local+global)
  Field recon: temporal mixing THEN fiber-dim FFT (both stages)
  3 causal convolutions per block: Wilson local, Wilson global, field temporal


In [4]:
@torch.no_grad()
def estimate_loss(model, cfg):
    model.eval()
    results = {}
    for name, sd in [("train", train_data), ("val", val_data)]:
        tot_ce, tot_ok, tot_n, tot_sp = 0., 0, 0, 0.
        bces = [0.] * cfg.n_blocks
        for _ in range(cfg.eval_steps):
            b = get_batch(sd, cfg)
            logits, info = model(b)
            tgt = b[:, 1:]
            ce = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), tgt.reshape(-1))
            tot_ce += ce.item()
            tot_ok += (logits.argmax(-1) == tgt).sum().item()
            tot_n += tgt.numel()
            tot_sp += info["mean_sparsity"]
            for i, bl in enumerate(info["intermediate_logits"]):
                bces[i] += F.cross_entropy(bl.reshape(-1, cfg.vocab_size), tgt.reshape(-1)).item()
        n = cfg.eval_steps
        results[name] = {
            "ce": tot_ce/n, "acc": tot_ok/tot_n, "sparsity": tot_sp/n,
            "block_ces": [c/n for c in bces],
        }
    model.train()
    return results

optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.05)

def lr_lambda(step):
    if step < config.warmup_steps:
        return step / max(1, config.warmup_steps)
    progress = (step - config.warmup_steps) / max(1, config.max_steps - config.warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

history = {
    "step": [], "train_ce": [], "val_ce": [], "train_acc": [], "val_acc": [],
    "train_sparsity": [], "val_sparsity": [],
    "train_bpc": [], "val_bpc": [],
    "block_val_ces": [[] for _ in range(config.n_blocks)],
    "lr": [],
}

print("Training V11.2: Temporal + Fiber Field Reconstruction")
print(f"Steps: {config.max_steps}, Batch: {config.batch_size}, Seq: {config.seq_len}")
print(f"LR: {config.learning_rate} with {config.warmup_steps}-step cosine warmup")
print(f"Key changes from v11.1:")
print(f"  1. Field recon: temporal mixing FIRST, then fiber-dim FFT (both stages)")
print(f"  2. Wilson line: added output projection (remixes local+global scales)")
print(f"  3. 3 causal convolutions per block (Wilson local, Wilson global, field temporal)")
print(f"Previous results:")
print(f"  v11.1: BPC ~2.47 | Acc ~48% (lost temporal mixing → slower learning)")
print(f"  v11.0: BPC ~2.34 | Acc ~53% | 9.3M params")
print(f"  nanoGPT: BPC 2.12 | CE 1.47 | ~10M params (target)")
print("=" * 70)

model.train()
pbar = tqdm(range(config.max_steps), desc="Training", unit="step")

for step in pbar:
    if step % config.eval_interval == 0:
        res = estimate_loss(model, config)
        tr, vl = res["train"], res["val"]
        history["step"].append(step)
        for k in ["ce", "acc", "sparsity"]:
            history[f"train_{k}"].append(tr[k])
            history[f"val_{k}"].append(vl[k])
        history["train_bpc"].append(tr["ce"] / math.log(2))
        history["val_bpc"].append(vl["ce"] / math.log(2))
        history["lr"].append(scheduler.get_last_lr()[0])
        for i in range(config.n_blocks):
            history["block_val_ces"][i].append(vl["block_ces"][i])
        tqdm.write(
            f"Step {step:5d} | Train CE: {tr['ce']:.3f} | Val CE: {vl['ce']:.3f} | "
            f"Val BPC: {vl['ce']/math.log(2):.2f} | Val Acc: {vl['acc']:.1%} | "
            f"Sp: {vl['sparsity']:.1%} | LR: {scheduler.get_last_lr()[0]:.2e}"
        )

    batch = get_batch(train_data, config)
    optimizer.zero_grad()
    logits, info = model(batch)
    targets = batch[:, 1:]

    ce_loss = sum(
        w * F.cross_entropy(bl.reshape(-1, config.vocab_size), targets.reshape(-1))
        for bl, w in zip(info["intermediate_logits"], info["block_loss_weights"])
    ) / info["block_loss_weights"].sum()

    dcl, nd = 0., 0
    for blk in model.blocks:
        for d in blk.memory.dictionaries:
            Dn = F.normalize(d, dim=-1)
            g = Dn @ Dn.T
            dcl += (g - torch.eye(g.size(0), device=g.device)).pow(2).mean()
            nd += 1
    dcl /= max(nd, 1)

    loss = ce_loss + 0.1 * dcl
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    with torch.no_grad():
        batch_acc = (logits.detach().argmax(-1) == targets).float().mean().item()
    pbar.set_postfix(
        ce=f"{ce_loss.item():.3f}", acc=f"{batch_acc:.1%}",
        bpc=f"{ce_loss.item()/math.log(2):.2f}", lr=f"{scheduler.get_last_lr()[0]:.1e}",
    )

res = estimate_loss(model, config)
tr, vl = res["train"], res["val"]
history["step"].append(config.max_steps)
for k in ["ce", "acc", "sparsity"]:
    history[f"train_{k}"].append(tr[k])
    history[f"val_{k}"].append(vl[k])
history["train_bpc"].append(tr["ce"] / math.log(2))
history["val_bpc"].append(vl["ce"] / math.log(2))
history["lr"].append(scheduler.get_last_lr()[0])
for i in range(config.n_blocks):
    history["block_val_ces"][i].append(vl["block_ces"][i])

print("=" * 70)
print(f"FINAL | Val CE: {vl['ce']:.3f} | Val BPC: {vl['ce']/math.log(2):.2f} | "
      f"Val Acc: {vl['acc']:.1%} | Val PPL: {math.exp(vl['ce']):.1f}")

Training V11.2: Temporal + Fiber Field Reconstruction
Steps: 20000, Batch: 8, Seq: 512
LR: 0.0003 with 2000-step cosine warmup
Key changes from v11.1:
  1. Field recon: temporal mixing FIRST, then fiber-dim FFT (both stages)
  2. Wilson line: added output projection (remixes local+global scales)
  3. 3 causal convolutions per block (Wilson local, Wilson global, field temporal)
Previous results:
  v11.1: BPC ~2.47 | Acc ~48% (lost temporal mixing → slower learning)
  v11.0: BPC ~2.34 | Acc ~53% | 9.3M params
  nanoGPT: BPC 2.12 | CE 1.47 | ~10M params (target)


Training:   0%|          | 0/20000 [00:00<?, ?step/s]

Step     0 | Train CE: 4.198 | Val CE: 4.198 | Val BPC: 6.06 | Val Acc: 0.9% | Sp: 75.0% | LR: 0.00e+00
Step   500 | Train CE: 2.320 | Val CE: 2.347 | Val BPC: 3.39 | Val Acc: 32.4% | Sp: 75.0% | LR: 7.50e-05
Step  1000 | Train CE: 1.984 | Val CE: 2.057 | Val BPC: 2.97 | Val Acc: 39.2% | Sp: 75.0% | LR: 1.50e-04
Step  1500 | Train CE: 1.841 | Val CE: 1.950 | Val BPC: 2.81 | Val Acc: 42.1% | Sp: 75.0% | LR: 2.25e-04
Step  2000 | Train CE: 1.726 | Val CE: 1.899 | Val BPC: 2.74 | Val Acc: 43.4% | Sp: 75.0% | LR: 3.00e-04
Step  2500 | Train CE: 1.646 | Val CE: 1.803 | Val BPC: 2.60 | Val Acc: 46.3% | Sp: 75.0% | LR: 2.99e-04
Step  3000 | Train CE: 1.588 | Val CE: 1.744 | Val BPC: 2.52 | Val Acc: 48.0% | Sp: 75.0% | LR: 2.98e-04
Step  3500 | Train CE: 1.566 | Val CE: 1.702 | Val BPC: 2.45 | Val Acc: 49.1% | Sp: 75.0% | LR: 2.95e-04
Step  4000 | Train CE: 1.507 | Val CE: 1.699 | Val BPC: 2.45 | Val Acc: 49.3% | Sp: 75.0% | LR: 2.91e-04
Step  4500 | Train CE: 1.469 | Val CE: 1.679 | Val BPC: 

KeyboardInterrupt: 

In [ ]:
steps = history["step"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].plot(steps, history["train_ce"], 'b-', label='Train')
axes[0,0].plot(steps, history["val_ce"], 'r-', label='Val')
axes[0,0].set_title('Cross-Entropy'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(steps, history["train_bpc"], 'b-', label='Train')
axes[0,1].plot(steps, history["val_bpc"], 'r-', label='Val')
axes[0,1].axhline(y=2.65, color='gray', linestyle='--', alpha=0.7, label='v9 (2.65)')
axes[0,1].axhline(y=3.50, color='green', linestyle='-.', alpha=0.7, label='v10.3 (3.50)')
axes[0,1].set_title('Bits per Character'); axes[0,1].legend(fontsize=8); axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(steps, history["train_acc"], 'b-', label='Train')
axes[0,2].plot(steps, history["val_acc"], 'r-', label='Val')
axes[0,2].axhline(y=0.45, color='gray', linestyle='--', alpha=0.7, label='v9 (45%)')
axes[0,2].axhline(y=0.296, color='green', linestyle='-.', alpha=0.7, label='v10.3 (30%)')
axes[0,2].set_title('Accuracy'); axes[0,2].legend(fontsize=8); axes[0,2].grid(True, alpha=0.3)

axes[1,0].plot(steps, history["lr"], 'purple', lw=2)
axes[1,0].set_title('Learning Rate'); axes[1,0].grid(True, alpha=0.3)

colors = plt.cm.viridis(np.linspace(0.2, 0.9, config.n_blocks))
for i in range(config.n_blocks):
    axes[1,1].plot(steps, history["block_val_ces"][i], color=colors[i], label=f'Block {i}')
axes[1,1].set_title('Per-Block Val CE'); axes[1,1].legend(fontsize=8); axes[1,1].grid(True, alpha=0.3)

# Sparsity per block output
with torch.no_grad():
    test_batch = get_batch(val_data, config)
    x_test, _ = model.embedding(test_batch)
    block_sparsities = []
    for blk in model.blocks:
        x_test = blk(x_test)
        block_sparsities.append((x_test.abs() < 1e-6).float().mean().item())

    axes[1,2].bar(range(config.n_blocks), block_sparsities, color=colors, edgecolor='black')
    axes[1,2].set_xlabel('Block'); axes[1,2].set_ylabel('Fraction of zeros')
    axes[1,2].set_title('Output Sparsity per Block\n(should be ~75% if re-sparsification works)')
    axes[1,2].set_ylim(0, 1); axes[1,2].grid(True, alpha=0.3, axis='y')

plt.suptitle('V11.1: True Fiber-Dim Field Reconstruction — Training Diagnostics',
             fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"\nFinal: Val BPC {history['val_bpc'][-1]:.2f} | Val Acc {history['val_acc'][-1]:.1%}")
print(f"v10.3: BPC 3.50 | Acc 30% (plateau at step 600)")
print(f"v9:    BPC 2.65 | Acc 45%")

In [ ]:
@torch.no_grad()
def generate_text(model, prompt_str, max_new=200, temperature=0.8):
    model.eval()
    cfg = model.cfg
    ids = torch.tensor(encode(prompt_str), dtype=torch.long).unsqueeze(0).to(device)
    for _ in range(max_new):
        ctx = ids[:, -cfg.max_seq_len:] if ids.size(1) >= cfg.max_seq_len else ids
        logits, _ = model(ctx)
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        ids = torch.cat([ids, torch.multinomial(probs, 1)], dim=1)
    return decode(ids[0].tolist())

print("=" * 60)
print("TEXT GENERATION (V11, temperature=0.8)")
print("=" * 60)
for p in ["ROMEO:\n", "To be or not to ", "The king ", "O, "]:
    print(f"\n--- Prompt: {repr(p)} ---")
    print(generate_text(model, p))

In [ ]:
@torch.no_grad()
def v11_diagnostics(model, cfg):
    """V11-specific diagnostics: field reconstruction, gauge rotation, sparsity evolution."""
    model.eval()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # --- Test 1: Does the field fill in unknowns? ---
    test_batch = get_batch(val_data, cfg)
    x_sparse, support_mask = model.embedding(test_batch)
    source_sparsity = (x_sparse.abs() < 1e-6).float().mean().item()

    # Trace through block 0 to get the field
    h = model.blocks[0].norm1(x_sparse)
    ctx = model.blocks[0].wilson(h)
    field = model.blocks[0].reconstructor(h, ctx)
    field_sparsity = (field.abs() < 1e-6).float().mean().item()

    axes[0,0].bar(['Sources', 'Field'], [source_sparsity, field_sparsity],
                  color=['steelblue', 'coral'], edgecolor='black')
    axes[0,0].set_ylabel('Fraction of zeros')
    axes[0,0].set_title('Sparsity: Sources vs Reconstructed Field\n'
                        '(field should be much denser)')
    axes[0,0].set_ylim(0, 1); axes[0,0].grid(True, alpha=0.3, axis='y')

    # --- Test 2: Block output sparsity + diffusion coefficient magnitude ---
    x = x_sparse
    block_sparsities = []
    diff_coeff_means = []
    for blk in model.blocks:
        h = blk.norm1(x)
        ctx = blk.wilson(h)
        dc = F.softplus(blk.reconstructor.D_proj(ctx))
        diff_coeff_means.append(dc.mean().item())
        x = blk(x)
        block_sparsities.append((x.abs() < 1e-6).float().mean().item())

    colors_v = plt.cm.viridis(np.linspace(0.2, 0.9, cfg.n_blocks))
    ax1 = axes[0,1]
    x_pos = np.arange(cfg.n_blocks)
    bars1 = ax1.bar(x_pos - 0.2, block_sparsities, 0.35, label='Output sparsity',
                    color=colors_v, edgecolor='black')
    ax2 = ax1.twinx()
    bars2 = ax2.bar(x_pos + 0.2, diff_coeff_means, 0.35, label='Mean D coeff',
                    color='coral', edgecolor='black', alpha=0.7)
    ax1.set_xlabel('Block'); ax1.set_ylabel('Sparsity (fraction zeros)')
    ax2.set_ylabel('Mean diffusion coefficient')
    ax1.set_title('Per-Block: Sparsity & Diffusion Strength')
    ax1.legend(loc='upper left', fontsize=7); ax2.legend(loc='upper right', fontsize=7)
    ax1.grid(True, alpha=0.3, axis='y')

    # --- Test 3: Langevin sparsity trajectory ---
    # Re-run block 0 Langevin step by step to track sparsity
    h = model.blocks[0].norm1(x_sparse)
    ctx = model.blocks[0].wilson(h)
    field = model.blocks[0].reconstructor(h, ctx)
    M_list = model.blocks[0].memory(field, h)

    step_sparsities = []
    x_lang = field.clone()
    betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps)
    sd = cfg.subbundle_dim
    K = cfg.n_subbundles
    BT = x_lang.shape[0] * x_lang.shape[1]
    M_all = torch.stack(M_list, dim=1)
    for step in range(cfg.langevin_steps):
        beta = betas[step].item()
        x_subs = x_lang.reshape(BT, K, sd)
        sim = torch.einsum('bks,bkas->bka', x_subs, M_all)
        w = F.softmax(beta * sim, dim=-1)
        grad = -torch.einsum('bka,bkas->bks', w, M_all)
        x_lang = x_lang - cfg.langevin_lr * grad.reshape_as(x_lang)
        if cfg.sparsity_lambda > 0:
            x_subs = x_lang.reshape(*x_lang.shape[:-1], K, sd)
            rms = x_subs.norm(dim=-1, keepdim=True) / math.sqrt(sd)
            thr = cfg.sparsity_lambda * rms
            x_lang = differentiable_proximal(x_subs, thr).reshape_as(x_lang)
        step_sparsities.append((x_lang.abs() < 1e-6).float().mean().item())

    axes[0,2].plot(range(cfg.langevin_steps), step_sparsities, 'b-o', lw=2, ms=6)
    axes[0,2].axhline(y=source_sparsity, color='gray', linestyle='--', alpha=0.7,
                      label=f'Source sparsity ({source_sparsity:.1%})')
    axes[0,2].set_xlabel('Langevin Step'); axes[0,2].set_ylabel('Fraction of zeros')
    axes[0,2].set_title('Sparsity Through Langevin Steps\n(should progressively increase)')
    axes[0,2].legend(); axes[0,2].grid(True, alpha=0.3)

    # --- Test 4: Wilson line context divergence ---
    text_a = "The brave knight drew his sword and charged into battle"
    text_b = "The quiet child drew his picture and smiled at mother"
    T = min(cfg.seq_len, len(text_a), len(text_b))
    ids_a = torch.tensor(encode(text_a[:T]), dtype=torch.long).unsqueeze(0).to(device)
    ids_b = torch.tensor(encode(text_b[:T]), dtype=torch.long).unsqueeze(0).to(device)

    xa, _ = model.embedding(ids_a)
    xb, _ = model.embedding(ids_b)
    ha = model.blocks[0].norm1(xa)
    hb = model.blocks[0].norm1(xb)
    ctx_a = model.blocks[0].wilson(ha)
    ctx_b = model.blocks[0].wilson(hb)
    ctx_div = (ctx_a - ctx_b).norm(dim=-1)[0].cpu().numpy()

    axes[1,0].plot(ctx_div, 'b-', lw=2)
    axes[1,0].set_xlabel('Position'); axes[1,0].set_ylabel('||ctx_a - ctx_b||')
    axes[1,0].set_title('Wilson Line Context Divergence\n(should grow as histories diverge)')
    axes[1,0].grid(True, alpha=0.3)

    # --- Test 5: Wilson line decay rate distributions (local vs global) ---
    for i, blk in enumerate(model.blocks):
        local_decay = F.softplus(blk.wilson.log_decay_local).cpu().numpy()
        global_decay = F.softplus(blk.wilson.log_decay_global).cpu().numpy()
        axes[1,1].hist(local_decay, bins=20, alpha=0.3, label=f'Local {i}')
        axes[1,2].hist(global_decay, bins=20, alpha=0.3, label=f'Global {i}')

    axes[1,1].set_xlabel('Decay rate'); axes[1,1].set_ylabel('Count')
    axes[1,1].set_title('Wilson Line LOCAL Decay Rates\n(fast decay = local context)')
    axes[1,1].legend(fontsize=7); axes[1,1].grid(True, alpha=0.3)

    axes[1,2].set_xlabel('Decay rate'); axes[1,2].set_ylabel('Count')
    axes[1,2].set_title('Wilson Line GLOBAL Decay Rates\n(slow decay = long-range context)')
    axes[1,2].legend(fontsize=7); axes[1,2].grid(True, alpha=0.3)

    plt.suptitle('V11 Diagnostics: Field Reconstruction + Geometry',
                 fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"Source sparsity: {source_sparsity:.1%}")
    print(f"Field sparsity:  {field_sparsity:.1%}")
    print(f"Langevin final sparsity: {step_sparsities[-1]:.1%}")
    print(f"Block output sparsities: {[f'{s:.1%}' for s in block_sparsities]}")
    print(f"Diffusion coefficients: {[f'Block {i}: {d:.3f}' for i, d in enumerate(diff_coeff_means)]}")

v11_diagnostics(model, config)

## Summary

### V11.2: Temporal + Fiber — Both Stages

**v11.1 correctly added fiber-dim reconstruction but accidentally removed temporal mixing.**

The field reconstructor's causal_exp_conv (temporal, along dim=1) was replaced with
per-subbundle FFT (fiber, along dim=-1). This was the right idea — fill in unknown
fiber dimensions — but it eliminated the only cross-position mechanism besides the
Wilson line. Result: slower learning than v11.0.

**v11.2 does both:**

1. **Field reconstruction stage 1: temporal mixing** — causal exponential decay
   convolution across positions (same as v11.0 had, now explicit). Each position
   gets a weighted history of all prior positions. This is the heat equation
   running along the sequence.

2. **Field reconstruction stage 2: fiber reconstruction** — per-subbundle FFT
   along the fiber dimension with context-dependent D and A. Fills in the 75%
   unknown dimensions from the temporally-enriched signal.

3. **Wilson line: input + output projections** — output proj remixes local and
   global temporal scales so the manifold can learn which scale combinations
   matter at each fiber dimension.

3 causal convolutions per block: Wilson local, Wilson global, field temporal.

### Architecture Flow

```
sparse sources (25% active per subbundle)
  → Wilson line: proj_in → dual-scale causal conv → proj_out (context)
  → field reconstructor:
      temporal conv (heat equation across positions) →
      per-subbundle fiber FFT (fill unknowns, context-warped D & A)
  → memory routing: field selects local attractors
  → Langevin settling: field → sparse (adaptive proximal every step)
  → residual + FFN → re-sparsify
  → sparse output → next block
```

### Architecture Comparison

| | v11.0 | v11.1 | v11.2 | nanoGPT |
|---|---|---|---|---|
| Temporal mixing | 1x causal conv (in field recon) | Wilson line only | Wilson (2-scale) + field conv | Multi-head attention |
| Fiber recon | None (mislabeled temporal) | Per-sub FFT (no temporal) | Temporal THEN per-sub FFT | N/A (dense) |
| Wilson params/block | 512 | ~263K | ~527K | N/A |
| Output sparsity | 0% | ~75% | ~75% | 0% (dense) |
| BPC | ~2.34 | ~2.47 | TBD | 2.12 |